In [1]:
import glob

import numpy as np
from sklearn.metrics import cohen_kappa_score, f1_score
from tabulate import tabulate

score_order = ["Wake", "N1", "N2", "N3", "REM"]

In [2]:
anphy_path = "../../examples/output"
anphy_preds_anysleep_glob = f"{anphy_path}/anphy/predictions_anysleep.npz"
anphy_preds_anysleep_files = sorted(glob.glob(anphy_preds_anysleep_glob))
print(anphy_preds_anysleep_files)

anphy_preds_anysleep_per_run = {}
for pred_file in anphy_preds_anysleep_files:
    f_handle = np.load(pred_file)
    keys = f_handle.files

    for k in keys:
        model_run = k.split("#")[1]
        s_id = k.split("#")[0].split(" ")[0]
        if model_run not in anphy_preds_anysleep_per_run:
            anphy_preds_anysleep_per_run[model_run] = {}
        anphy_preds_anysleep_per_run[model_run][s_id] = f_handle[k]

anphy_preds_usleep_glob = f"{anphy_path}/anphy/predictions_usleep_c.npz"
anphy_preds_usleep_files = sorted(glob.glob(anphy_preds_usleep_glob))
print(anphy_preds_usleep_files)

anphy_preds_usleep_per_run = {}
for pred_file in anphy_preds_usleep_files:
    f_handle = np.load(pred_file)
    keys = f_handle.files

    for k in keys:
        model_run = k.split("#")[1]
        s_id = k.split("#")[0].split(" ")[0]
        if model_run not in anphy_preds_usleep_per_run:
            anphy_preds_usleep_per_run[model_run] = {}
        anphy_preds_usleep_per_run[model_run][s_id] = f_handle[k]

# FIXME: add path to ground truth annotation files of ANPHY dataset
anphy_gt_glob = f"<path-to-anphy-annotation-files>/*.txt"
anphy_gt_files = sorted(glob.glob(anphy_gt_glob))

anphy_gt_ss = {}
for gt_file in anphy_gt_files:
    s_id = gt_file.split("/")[-1].split(".")[0]
    print(f"Processing {s_id}...")
    with open(gt_file, "r") as f:
        f_content = f.readlines()

    sleep_stages = []
    p_line = None
    for line in f_content:
        tokens = line.strip().split("\t")
        if p_line is None and tokens[1] != "0":
            n_to_add = int(int(tokens[1]) / 30)
            print(f"adding {n_to_add} epochs to the start")
            sleep_stages += ["UNK" for _ in range(n_to_add)]
        sleep_stages.append(tokens[0])
        assert tokens[2] == "30", tokens

        # check that there is no time gap to the previous annotation
        if p_line is not None:
            p_tokens = p_line.split("\t")
            c_time = int(tokens[1])
            p_time = int(p_tokens[1])
            if c_time - p_time > 30:
                n_to_add = int((c_time - p_time) / 30) - 1
                print(f"adding {n_to_add} epochs")
                sleep_stages += [p_tokens[0] for _ in range(n_to_add)]
        p_line = line
    anphy_gt_ss[s_id] = sleep_stages

# map sleep stages to numbers
stage_map = {
    "W": 0, "N1": 1, "N2": 2, "N3": 3, "R": 4, "L": 9, "UNK": 9
}
anphy_gt_ss_num = {s_id: [stage_map[s] for s in v] for s_id, v in anphy_gt_ss.items()}

['../../examples/output/anphy/predictions_anysleep.npz']
['../../examples/output/anphy/predictions_usleep_c.npz']
Processing EPCTL01...
Processing EPCTL02...
Processing EPCTL03...
Processing EPCTL04...
Processing EPCTL05...
Processing EPCTL06...
Processing EPCTL07...
Processing EPCTL09...
Processing EPCTL10...
Processing EPCTL11...
Processing EPCTL12...
Processing EPCTL13...
Processing EPCTL14...
Processing EPCTL15...
Processing EPCTL16...
Processing EPCTL17...
Processing EPCTL18...
Processing EPCTL19...
Processing EPCTL20...
Processing EPCTL21...
Processing EPCTL22...
Processing EPCTL23...
adding 1 epochs to the start
Processing EPCTL24...
Processing EPCTL25...
Processing EPCTL26...
Processing EPCTL27...
Processing EPCTL28...
Processing EPCTL29...


In [3]:
anphy_all_stages_pred_anysleep = {mr: [] for mr in anphy_preds_anysleep_per_run}
anphy_all_stages_pred_usleep = {mr: [] for mr in anphy_preds_usleep_per_run}
anphy_all_stages_gt = []

for s_id in anphy_gt_ss_num:
    for i, mr in enumerate(anphy_preds_anysleep_per_run):
        if len(anphy_gt_ss_num[s_id]) != len(anphy_preds_anysleep_per_run[mr][s_id]):
            print(
                f"Length mismatch in s_id {s_id} with lengths {len(anphy_gt_ss_num[s_id])} and {len(anphy_preds_anysleep_per_run[mr][s_id])}"
                f", removing {len(anphy_gt_ss_num[s_id]) - len(anphy_preds_anysleep_per_run[mr][s_id])} epochs from GT")
            gt = anphy_gt_ss_num[s_id][:len(anphy_preds_anysleep_per_run[mr][s_id])]
        else:
            gt = anphy_gt_ss_num[s_id]
        if i == 0:
            anphy_all_stages_gt.extend(gt)
        anphy_all_stages_pred_anysleep[mr].extend(anphy_preds_anysleep_per_run[mr][s_id].tolist())

    for i, mr in enumerate(anphy_preds_usleep_per_run):
        if len(anphy_gt_ss_num[s_id]) != len(anphy_preds_usleep_per_run[mr][s_id]):
            print(
                f"Length mismatch in s_id {s_id} with lengths {len(anphy_gt_ss_num[s_id])} and {len(anphy_preds_usleep_per_run[mr][s_id])}"
                f", removing {len(anphy_gt_ss_num[s_id]) - len(anphy_preds_usleep_per_run[mr][s_id])} epochs from GT")
        anphy_all_stages_pred_usleep[mr].extend(anphy_preds_usleep_per_run[mr][s_id].tolist())

anphy_all_stages_gt = np.array(anphy_all_stages_gt)
anphy_all_stages_pred_anysleep = {mr: np.array(anphy_all_stages_pred_anysleep[mr]) for mr in
                                  anphy_all_stages_pred_anysleep}
anphy_all_stages_pred_usleep = {mr: np.array(anphy_all_stages_pred_usleep[mr]) for mr in anphy_all_stages_pred_usleep}

Length mismatch in s_id EPCTL01 with lengths 958 and 957, removing 1 epochs from GT
Length mismatch in s_id EPCTL01 with lengths 958 and 957, removing 1 epochs from GT
Length mismatch in s_id EPCTL01 with lengths 958 and 957, removing 1 epochs from GT
Length mismatch in s_id EPCTL01 with lengths 958 and 957, removing 1 epochs from GT
Length mismatch in s_id EPCTL01 with lengths 958 and 957, removing 1 epochs from GT
Length mismatch in s_id EPCTL01 with lengths 958 and 957, removing 1 epochs from GT
Length mismatch in s_id EPCTL02 with lengths 1073 and 1072, removing 1 epochs from GT
Length mismatch in s_id EPCTL02 with lengths 1073 and 1072, removing 1 epochs from GT
Length mismatch in s_id EPCTL02 with lengths 1073 and 1072, removing 1 epochs from GT
Length mismatch in s_id EPCTL02 with lengths 1073 and 1072, removing 1 epochs from GT
Length mismatch in s_id EPCTL02 with lengths 1073 and 1072, removing 1 epochs from GT
Length mismatch in s_id EPCTL02 with lengths 1073 and 1072, removi

In [4]:
f1s_usleep = []
f1s_anysleep = []
cohks_usleep = {}
cohks_anysleep = {}

for mr in anphy_all_stages_pred_usleep:
    art_mask = anphy_all_stages_gt == 9
    f1s_usleep.append(
        f1_score(anphy_all_stages_gt[~art_mask], anphy_all_stages_pred_usleep[mr][~art_mask], labels=range(5),
                 average=None).tolist())
    f1s_usleep[-1].append(np.mean(f1s_usleep[-1]))

    cohk = cohen_kappa_score(anphy_all_stages_gt[~art_mask], anphy_all_stages_pred_usleep[mr][~art_mask])
    cohks_usleep[mr + ".pth\n"] = cohk

for mr in anphy_all_stages_pred_anysleep:
    print(mr)
    art_mask = anphy_all_stages_gt == 9
    f1s_anysleep.append(
        f1_score(anphy_all_stages_gt[~art_mask], anphy_all_stages_pred_anysleep[mr][~art_mask], labels=range(5),
                 average=None).tolist())
    f1s_anysleep[-1].append(np.mean(f1s_anysleep[-1]))

    cohk = cohen_kappa_score(anphy_all_stages_gt[~art_mask], anphy_all_stages_pred_anysleep[mr][~art_mask])
    cohks_anysleep[mr + ".pth\n"] = cohk

anysleep-run1
anysleep-run2
anysleep-run3


In [5]:
# present scores as table
table_data = np.empty((2, len(score_order) + 4), dtype=object)

# AnySleep
means = np.mean(f1s_anysleep, axis=0)
stds = np.std(f1s_anysleep, axis=0)
table_data[0, 2:-1] = [f"{m:.2f} ({s:.3f})" for m, s in zip(means, stds)]

cks = [cohks_anysleep[m] for m in cohks_anysleep.keys()]
table_data[0, -1] = f"{np.mean(cks):.2f} ({np.std(cks):.3f})"

# U-Sleep
means = np.mean(f1s_usleep, axis=0)
stds = np.std(f1s_usleep, axis=0)
table_data[1, 2:-1] = [f"{m:.2f} ({s:.3f})" for m, s in zip(means, stds)]

cks = [cohks_usleep[m] for m in cohks_usleep.keys()]
table_data[1, -1] = f"{np.mean(cks):.2f} ({np.std(cks):.3f})"

table_data[:, 0] = ["AnySleep", "U-Sleep"]
table_data[:, 1] = [28, 28]
table_data[:, 1] = [str(t) for t in table_data[:, 1]]

In [6]:
table_data = table_data.tolist()

tabulate(table_data,
         headers=["", ""] + score_order + ["MF1", "Cohen's Kappa"],
         tablefmt="unsafehtml")

,,Wake,N1,N2,N3,REM,MF1,Cohen's Kappa
AnySleep,28,0.91 (0.002),0.46 (0.009),0.90 (0.000),0.91 (0.005),0.88 (0.002),0.81 (0.002),0.82 (0.002)
U-Sleep,28,0.89 (0.010),0.47 (0.001),0.89 (0.005),0.87 (0.019),0.87 (0.002),0.80 (0.004),0.79 (0.006)


In [7]:
print(tabulate(table_data,
               headers=["", ""] + score_order + ["MF1", "Cohen's Kappa"],
               tablefmt="latex_raw"))

\begin{tabular}{lrlllllll}
\hline
          &    & Wake         & N1           & N2           & N3           & REM          & MF1          & Cohen's Kappa   \\
\hline
 AnySleep & 28 & 0.91 (0.002) & 0.46 (0.009) & 0.90 (0.000) & 0.91 (0.005) & 0.88 (0.002) & 0.81 (0.002) & 0.82 (0.002)    \\
 U-Sleep  & 28 & 0.89 (0.010) & 0.47 (0.001) & 0.89 (0.005) & 0.87 (0.019) & 0.87 (0.002) & 0.80 (0.004) & 0.79 (0.006)    \\
\hline
\end{tabular}
